In [1]:
%pip install torch

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [5]:
import torch
import torch.nn as nn

# 1. DEFINE THE ARCHITECTURE
class BatterySoCEstimator(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Layer 1: Takes our 3 sensor inputs and expands them into 16 "neurons"
        self.layer1 = nn.Linear(in_features=3, out_features=16) 
        
        # Layer 2: Compresses those 16 neurons down into 8 neurons
        self.layer2 = nn.Linear(in_features=16, out_features=8)
        
        # Output Layer: Takes the 8 neurons and outputs 1 single continuous number (SoC)
        self.output_layer = nn.Linear(in_features=8, out_features=1)
        
        # Activation Function: The "ReLU" introduces non-linear physics. 
        # Without this, the network could only draw straight lines.
        self.relu = nn.ReLU()

    # 2. DEFINE THE WIRING (How the data flows from input to output)
    def forward(self, x):
        # Data 'x' goes into layer 1, then gets passed through the ReLU activation
        x = self.relu(self.layer1(x))
        
        # Then it goes into layer 2, and gets activated again
        x = self.relu(self.layer2(x))
        
        # Finally, it goes to the output layer. (No activation here because we want a raw number)
        soc_estimate = self.output_layer(x)
        
        return soc_estimate

print("✅ Neural Network Architecture Defined!")

✅ Neural Network Architecture Defined!


In [6]:
# 1. BUILD THE NETWORK
# This takes the class we just defined and instantiates it in memory
model = BatterySoCEstimator()

# 2. CREATE A DUMMY CAN FRAME (The Input)
# Let's simulate: 390.0 Volts, -50.0 Amps (Discharging), 25.0 Celsius
# Notice the double brackets [[ ]] - this makes it a 2D Tensor (Batch Size of 1)
dummy_sensor_data = torch.tensor([[390.0, -50.0, 25.0]])

print("--- The Input Signal ---")
print(f"Voltage, Current, Temp: {dummy_sensor_data.tolist()}\n")

# 3. PUSH THE SIGNAL THROUGH THE NETWORK (The Forward Pass)
# We don't need to call model.forward(), PyTorch handles that automatically
untrained_soc_estimate = model(dummy_sensor_data)

print("--- The Untrained Output ---")
# .item() just pulls the single raw number out of the PyTorch Tensor format
print(f"Predicted State of Charge: {untrained_soc_estimate.item():.2f}%")

--- The Input Signal ---
Voltage, Current, Temp: [[390.0, -50.0, 25.0]]

--- The Untrained Output ---
Predicted State of Charge: 70.85%


In [7]:
import torch.optim as optim

# 1. DEFINE THE LOSS FUNCTION AND OPTIMIZER
# MSE measures how far off our predicted percentage is from the true percentage
criterion = nn.MSELoss()

# Adam optimizer looks at the model's parameters (the 16 neurons) and tweaks them.
# lr=0.01 is the "Learning Rate" (how big of an adjustment it makes per step).
optimizer = optim.Adam(model.parameters(), lr=0.01)

# 2. GENERATE SYNTHETIC TRAINING DATA
# Let's create 100 simulated CAN frames: [Voltage, Current, Temp]
# Voltage between 300V-400V, Current between -100A to 100A, Temp 10C to 40C
voltages = (torch.rand(100, 1) * 100) + 300
currents = (torch.rand(100, 1) * 200) - 100
temps = (torch.rand(100, 1) * 30) + 10

X_train = torch.cat((voltages, currents, temps), dim=1)

# Let's define a "True Physics" rule for the network to learn:
# SoC is heavily dependent on Voltage, drops slightly under heavy Current load, and drops in cold Temps.
# (This is a simplified physics model just for the AI to practice on)
y_train = (voltages - 300) * 0.8 - (currents * 0.05) + (temps * 0.2) 

# 3. THE ACTUAL TRAINING LOOP
epochs = 500
print("⚙️ Starting Neural Network Calibration...\n")

for epoch in range(epochs):
    # Step A: Forward Pass (Make a prediction on all 100 frames at once)
    predictions = model(X_train)
    
    # Step B: Calculate the Error (How far off were we from the true SoC?)
    loss = criterion(predictions, y_train)
    
    # Step C: Zero out the old gradients (Clear the memory from the last loop)
    optimizer.zero_grad()
    
    # Step D: Backward Pass (Calculate exactly which neurons caused the error)
    loss.backward()
    
    # Step E: Optimizer Step (Tweak the weights to fix the error)
    optimizer.step()
    
    # Print progress every 100 epochs
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Mean Squared Error: {loss.item():.2f}")

print("\n✅ Calibration Complete!")

⚙️ Starting Neural Network Calibration...

Epoch 100/500 | Mean Squared Error: 342.78
Epoch 200/500 | Mean Squared Error: 334.64
Epoch 300/500 | Mean Squared Error: 329.40
Epoch 400/500 | Mean Squared Error: 321.76
Epoch 500/500 | Mean Squared Error: 308.17

✅ Calibration Complete!


In [8]:
# 1. THE VALIDATION FRAME (Same as before)
dummy_sensor_data = torch.tensor([[390.0, -50.0, 25.0]])

# 2. PUSH THROUGH THE TRAINED NETWORK
# We use torch.no_grad() to tell PyTorch we are just testing, not training anymore
with torch.no_grad():
    trained_soc_estimate = model(dummy_sensor_data)

print("--- The Trained Output ---")
print(f"Predicted State of Charge: {trained_soc_estimate.item():.2f}%")

# 3. CALCULATE THE "TRUE" PHYSICS TARGET
# We plug the dummy data into the exact rule we used to train it
true_soc = (390.0 - 300) * 0.8 - (-50.0 * 0.05) + (25.0 * 0.2)
print(f"True Physical State of Charge: {true_soc:.2f}%")
print(f"Absolute Error: {abs(true_soc - trained_soc_estimate.item()):.2f}%")

--- The Trained Output ---
Predicted State of Charge: 66.25%
True Physical State of Charge: 79.50%
Absolute Error: 13.25%


In [9]:
# 1. THE TWEAKED OPTIMIZER (Smaller, more precise steps)
optimizer = optim.Adam(model.parameters(), lr=0.002)

# 2. MORE DYNO TIME (Increase loops from 500 to 5000)
epochs = 5000

print("⚙️ Restarting Neural Network Calibration...\n")

for epoch in range(epochs):
    predictions = model(X_train)
    loss = criterion(predictions, y_train)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # Print progress every 1000 epochs to keep the terminal clean
    if (epoch + 1) % 1000 == 0:
        print(f"Epoch {epoch+1}/{epochs} | Mean Squared Error: {loss.item():.2f}")

print("\n✅ Recalibration Complete!")

# 3. INSTANT VALIDATION TEST
with torch.no_grad():
    recalibrated_estimate = model(dummy_sensor_data)

print("-" * 40)
print(f"True Physical State of Charge: {true_soc:.2f}%")
print(f"NEW Predicted State of Charge: {recalibrated_estimate.item():.2f}%")
print(f"NEW Absolute Error: {abs(true_soc - recalibrated_estimate.item()):.2f}%")
print("-" * 40)

⚙️ Restarting Neural Network Calibration...

Epoch 1000/5000 | Mean Squared Error: 97.29
Epoch 2000/5000 | Mean Squared Error: 3.72
Epoch 3000/5000 | Mean Squared Error: 0.99
Epoch 4000/5000 | Mean Squared Error: 3.36
Epoch 5000/5000 | Mean Squared Error: 0.00

✅ Recalibration Complete!
----------------------------------------
True Physical State of Charge: 79.50%
NEW Predicted State of Charge: 79.48%
NEW Absolute Error: 0.02%
----------------------------------------
